In [1]:
from pyspark.sql import SparkSession

spark = SparkSession. \
builder. \
appName("week9-lesson-6"). \
config("spark.sql.warehouse.dir", f"/user/itv027484/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()

### spark.sql.files.maxPartitionBytes  - Initial partitions in a Dataframe - reading a single file-splittable

In [2]:
# [itv027484@g02 ~]$ spark3-submit --master yarn --num-executors 2 --executor-cores 4 --executor-memory 1G --conf spark.dynamicAllocation.enabled=false pgm1.py
# SPARK_MAJOR_VERSION is set to 3, using Spark3
# 9
# +--------+
# |count(1)|
# +--------+
# | 2833500|
# +--------+

# [itv027484@g02 ~]$ spark3-submit --master yarn --num-executors 5 --executor-cores 4 --executor-memory 2G --conf spark.dynamicAllocation.enabled=false pgm1.py
# SPARK_MAJOR_VERSION is set to 3, using Spark3
# 20
# +--------+
# |count(1)|
# +--------+
# | 2833500|
# +--------+

#### number of partitions is determined by spark based on number of cores available and size of the file

###  partition size = min(maxPartitionBytes,file-size/defaultParallelism)

In [5]:
spark.conf.get("spark.sql.files.maxPartitionBytes")

'134217728b'

In [6]:
134217728 / (1024 *1024)

128.0

In [7]:
order_schema= 'order_id long, order_date string , customer_id long, order_status string'

In [8]:
orders_df = spark.read.format('csv').schema(order_schema).load('/public/trendytech/orders/orders_1gb.csv')

In [21]:
orders_df.rdd.getNumPartitions()

9

In [10]:
new_order_df = orders_df.repartition(1)

In [11]:
new_order_df.write \
.format('csv') \
.mode('overwrite') \
.option('codec','gzip') \
.save('orders_zip')

### saved as a zip file. non splittable

In [12]:
df = spark.read.format('csv').schema(order_schema).load('orders_zip')

In [13]:
df.show(3)

+--------+--------------------+-----------+---------------+
|order_id|          order_date|customer_id|   order_status|
+--------+--------------------+-----------+---------------+
|       1|2013-07-25 00:00:...|      11599|         CLOSED|
|       2|2013-07-25 00:00:...|        256|PENDING_PAYMENT|
|       3|2013-07-25 00:00:...|      12111|       COMPLETE|
+--------+--------------------+-----------+---------------+
only showing top 3 rows



### non splittable file - hence only 1 partition after reading  - Initial partitions in a Dataframe - reading a single file-NON splittable

In [22]:
df.rdd.getNumPartitions()

1

### saved as a snappy file. non splittable

In [15]:
new_order_df.write \
.format('csv') \
.mode('overwrite') \
.option('codec','snappy') \
.save('orders_snappy')

In [16]:
df1 = spark.read.format('csv').schema(order_schema).load('orders_snappy')

### non splittable file - hence only 1 partition after reading

In [23]:
df1.rdd.getNumPartitions()

1

### save as parquet - splittable even when compressed by snappy

In [33]:
new_order_df.write \
.mode('overwrite') \
.save('orders_pq')

In [34]:
df2 = spark.read.load('orders_pq')

### getNumPartitions 
#### When creating an RDD or DataFrame from local collections or reading files without explicit partitioning, 
#### Spark defaults to the number of currently available CPU cores (spark.default.parallelism).

In [35]:
df2.rdd.getNumPartitions()

7

In [38]:
df3 = spark.read.load('orders_pq')

In [39]:
df3.rdd.getNumPartitions()

2

In [42]:
df4 = df3.repartition(20)

In [43]:
df4.write \
.mode('overwrite') \
.save('orders_pq20')

In [48]:
df5 = spark.read.load('orders_pq20')

In [49]:
df5.rdd.getNumPartitions()

2

### time take to open one file - spark.sql.files.openCostInBytes - equivalent to reading this much bytes of data

In [50]:
spark.conf.get('spark.sql.files.openCostInBytes')

'4194304'

In [51]:
spark.conf.get('spark.sql.adaptive.enabled')

'false'

In [52]:
spark.conf.get('spark.sql.adaptive.coalescePartitions.enabled')

'true'